In [23]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sqlite3
import os

# Plotting purposes
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (14, 5), "axes.titlesize": 14})

PLOT_DIR = "eda_plots_GOH"
os.makedirs(PLOT_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(PLOT_DIR, name), dpi=100, bbox_inches="tight")
    plt.close()
    print(f"   saved {PLOT_DIR}/{name}")

In [24]:
conn = sqlite3.connect("../data/driving_events.db")
df = pd.read_sql_query("SELECT * FROM SENSOR_TABLE", conn)
conn.close()

# Parse TIME into Hour, Minute, Second for sampling-rate analysis
time_parts = df["TIME"].str.split(":", expand=True).astype(int)
df["Hour"] = time_parts[0]
df["Minute"] = time_parts[1]
df["Second"] = time_parts[2]

print(df.head())
print("=" * 40)
print(f"Rows: {len(df)}")
print(f"Cols: {df.columns.tolist()}")
print(f"Missing:\n{df.isnull().sum()}")

       TIME   ACCEL_X   ACCEL_Y   ACCEL_Z    GYRO_X    GYRO_Y    GYRO_Z  \
0  06:45:33  0.213084  9.924265 -2.166753 -0.008934 -0.018097 -0.017104   
1  06:45:33  0.213084  9.924265 -2.166753 -0.008934 -0.018097 -0.017104   
2  06:45:33  0.213084  9.924265 -2.166753 -0.008934 -0.018097 -0.017104   
3  06:45:33  0.560243  8.806173 -2.741361 -0.008934 -0.018097 -0.017104   
4  06:45:33  0.193930  9.344869 -1.414973  0.036270 -0.001604 -0.024435   

   CURRENT_SPEED ACTIVITY  Hour  Minute  Second  
0            0.0    Right     6      45      33  
1            0.0    Right     6      45      33  
2            0.0    Right     6      45      33  
3            0.0    Right     6      45      33  
4            0.0    Right     6      45      33  
Rows: 128512
Cols: ['TIME', 'ACCEL_X', 'ACCEL_Y', 'ACCEL_Z', 'GYRO_X', 'GYRO_Y', 'GYRO_Z', 'CURRENT_SPEED', 'ACTIVITY', 'Hour', 'Minute', 'Second']
Missing:
TIME             0
ACCEL_X          0
ACCEL_Y          0
ACCEL_Z          0
GYRO_X          

In [25]:
samples_per_second = df.groupby(["Hour", "Minute", "Second"]).size()
print("Samples per second distribution:")
print(samples_per_second.value_counts())
estimated_hz = samples_per_second.mode()[0]
print(f"\nEstimated Sampling Rate: {estimated_hz} Hz")

Samples per second distribution:
56    1096
55     901
12     178
10     141
57      40
54      38
37      12
53      12
19      11
35      10
32      10
2       10
43      10
39      10
11      10
48       9
7        9
16       9
52       9
28       9
23       9
36       8
44       8
24       8
51       8
31       8
6        8
9        7
50       7
4        7
22       6
42       6
27       6
45       6
3        6
47       6
8        6
33       5
20       5
41       5
40       5
5        5
49       5
30       5
34       5
18       4
14       4
1        4
38       4
26       4
29       4
13       3
46       3
15       3
17       2
21       2
25       1
Name: count, dtype: int64

Estimated Sampling Rate: 56 Hz


In [26]:
# Number of driving events
df.groupby(["ACTIVITY"]).size()

ACTIVITY
Accelerate               14036
Aggressive Accelerate    10897
Aggressive Brake         10447
Aggressive Left          14511
Aggressive Right         13886
Brake                    18631
Idling                    9321
Left                     18626
Right                    18157
dtype: int64

In [27]:
plt.figure(figsize=(20, 10))
plt.plot(df["ACCEL_X"].iloc[:100])
plt.title("ACCEL_X – first 100 samples")
plt.xlabel("Sample index")
plt.ylabel("ACCEL_X")
plt.show()

/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_23929/3438255598.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
# Schema inspection
print("\n— dtypes —")
print(df.dtypes)
print(f"\n— Memory usage —\n{df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"\n— Missing values —\n{df.isnull().sum()}")
print(f"\n— Duplicate rows: {df.duplicated().sum()}")
print(f"\n— Describe —")
print(df.describe())


— dtypes —
TIME                 str
ACCEL_X          float64
ACCEL_Y          float64
ACCEL_Z          float64
GYRO_X           float64
GYRO_Y           float64
GYRO_Z           float64
CURRENT_SPEED    float64
ACTIVITY             str
Hour               int64
Minute             int64
Second             int64
dtype: object

— Memory usage —
25.19 MB

— Missing values —
TIME             0
ACCEL_X          0
ACCEL_Y          0
ACCEL_Z          0
GYRO_X           0
GYRO_Y           0
GYRO_Z           0
CURRENT_SPEED    0
ACTIVITY         0
Hour             0
Minute           0
Second           0
dtype: int64

— Duplicate rows: 13928

— Describe —
             ACCEL_X        ACCEL_Y        ACCEL_Z         GYRO_X  \
count  128512.000000  128512.000000  128512.000000  128512.000000   
mean        0.047994       9.767517       0.046954      -0.000821   
std         0.960505       0.337302       1.192786       0.055620   
min        -5.466262       7.170933      -7.082348      -0.363236   
25

In [29]:
samples_per_second = df.groupby(["Hour", "Minute", "Second"]).size()

print("\nSamples-per-second distribution:")
print(samples_per_second.value_counts().sort_index())

estimated_hz = int(samples_per_second.mode()[0])
print(f"\n>>> Estimated sampling rate: {estimated_hz} Hz")

fig, ax = plt.subplots()
samples_per_second.hist(bins=30, ax=ax, edgecolor="black")
ax.set_xlabel("Samples per second")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Samples per Wall-Clock Second")
ax.axvline(estimated_hz, color="red", ls="--", label=f"Mode = {estimated_hz} Hz")
ax.legend()
plt.tight_layout()
savefig("01_sampling_rate.png")


Samples-per-second distribution:
1        4
2       10
3        6
4        7
5        5
6        8
7        9
8        6
9        7
10     141
11      10
12     178
13       3
14       4
15       3
16       9
17       2
18       4
19      11
20       5
21       2
22       6
23       9
24       8
25       1
26       4
27       6
28       9
29       4
30       5
31       8
32      10
33       5
34       5
35      10
36       8
37      12
38       4
39      10
40       5
41       5
42       6
43      10
44       8
45       6
46       3
47       6
48       9
49       5
50       7
51       8
52       9
53      12
54      38
55     901
56    1096
57      40
Name: count, dtype: int64

>>> Estimated sampling rate: 56 Hz
   saved eda_plots_GOH/01_sampling_rate.png


In [30]:
LABEL_COL = "ACTIVITY"
class_counts = df[LABEL_COL].value_counts().sort_index()
print(f"\nActivity counts:\n{class_counts}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts.plot.bar(ax=axes[0], edgecolor="black")
axes[0].set_title("Driving-Event Activity Counts")
axes[0].set_ylabel("Samples")
axes[0].set_xlabel("Activity")
axes[0].tick_params(axis="x", rotation=45)

class_counts.plot.pie(ax=axes[1], autopct="%1.1f%%", startangle=140)
axes[1].set_ylabel("")
axes[1].set_title("Activity Proportions")

plt.tight_layout()
savefig("02_class_distribution.png")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (max / min): {imbalance_ratio:.1f}x")


Activity counts:
ACTIVITY
Accelerate               14036
Aggressive Accelerate    10897
Aggressive Brake         10447
Aggressive Left          14511
Aggressive Right         13886
Brake                    18631
Idling                    9321
Left                     18626
Right                    18157
Name: count, dtype: int64
   saved eda_plots_GOH/02_class_distribution.png

Imbalance ratio (max / min): 2.0x


In [40]:
META_COLS = ["ACTIVITY", "TIME", "Hour", "Minute", "Second", "CURRENT_SPEED"]
LABEL_COL = "ACTIVITY"
sensor_cols = [c for c in df.columns if c not in META_COLS]
print(f"\nSensor channels ({len(sensor_cols)}): {sensor_cols}")
print(df[sensor_cols].describe().T)

# Per-channel histograms
n = len(sensor_cols)
ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_2d(axes)

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // ncols, idx % ncols]
    df[col].hist(bins=80, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel("Value")

for idx in range(n, nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

fig.suptitle("Sensor-Channel Value Distributions", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()
savefig("03_channel_histograms.png")


Sensor channels (7): ['ACCEL_X', 'ACCEL_Y', 'ACCEL_Z', 'GYRO_X', 'GYRO_Y', 'GYRO_Z', 'label_name']
            count      mean       std       min       25%       50%       75%  \
ACCEL_X  128512.0  0.047994  0.960505 -5.466262 -0.270545  0.074220  0.442927   
ACCEL_Y  128512.0  9.767517  0.337302  7.170933  9.567530  9.768642  9.976937   
ACCEL_Z  128512.0  0.046954  1.192786 -7.082348 -0.529119  0.064643  0.641646   
GYRO_X   128512.0 -0.000821  0.055620 -0.363236 -0.029703 -0.000535  0.029551   
GYRO_Y   128512.0  0.000723  0.253577 -3.779423 -0.027260  0.000000  0.028711   
GYRO_Z   128512.0  0.000100  0.026307 -3.089985 -0.013057 -0.000229  0.013439   

               max  
ACCEL_X   5.815815  
ACCEL_Y  12.423812  
ACCEL_Z   8.018480  
GYRO_X    6.849709  
GYRO_Y    0.870712  
GYRO_Z    0.176540  


/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_23929/2041833754.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


   saved eda_plots_GOH/03_channel_histograms.png


In [32]:
# Raw waveform plot (first 500 samples)
PREVIEW_LEN = 500

fig, axes = plt.subplots(len(sensor_cols), 1,
                         figsize=(16, 3 * len(sensor_cols)), sharex=True)
if len(sensor_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, sensor_cols):
    ax.plot(df[col].iloc[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"Raw Waveforms (first {PREVIEW_LEN} samples)", fontsize=15)
plt.tight_layout()
savefig("04_raw_waveforms.png")
plt.show()

   saved eda_plots_GOH/04_raw_waveforms.png


/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_23929/194218340.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
# Per-class signals — GYRO_X
PRIMARY_AXIS = "GYRO_X"
classes = sorted(df[LABEL_COL].unique())

fig, axes = plt.subplots(len(classes), 1,
                         figsize=(16, 3 * len(classes)), sharex=True, sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    subset = df.loc[df[LABEL_COL] == cls, PRIMARY_AXIS].values
    ax.plot(subset[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(PRIMARY_AXIS)
    ax.set_title(f"Activity: {cls}  (n={len(subset):,})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"{PRIMARY_AXIS} Waveform by Driving-Event Activity", fontsize=15)
plt.tight_layout()
savefig("05_per_class_GYRO_X.png")

   saved eda_plots_GOH/05_per_class_GYRO_X.png


In [34]:
# Per-class waveform overlay for ACCEL_X
PRIMARY_AXIS2 = "ACCEL_X"

fig, axes = plt.subplots(len(classes), 1,
                         figsize=(16, 3 * len(classes)), sharex=True, sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    subset = df.loc[df[LABEL_COL] == cls, PRIMARY_AXIS2].values
    ax.plot(subset[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(PRIMARY_AXIS2)
    ax.set_title(f"Activity: {cls}  (n={len(subset):,})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"{PRIMARY_AXIS2} Waveform by Driving-Event Activity", fontsize=15)
plt.tight_layout()
savefig("06_per_class_ACCEL_X.png")

   saved eda_plots_GOH/06_per_class_ACCEL_X.png


In [35]:
# Correlation & Cross-Channel Relationships

corr = df[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, ax=ax)
ax.set_title("Sensor-Channel Correlation Matrix")
plt.tight_layout()
savefig("07_correlation_heatmap.png")

# Pairplot — use Accel vs Gyro subset (4 channels) for speed
pair_cols = ["ACCEL_X", "ACCEL_Z", "GYRO_X", "GYRO_Y"]
LABEL_NAME = "label_name"
df[LABEL_NAME] = df[LABEL_COL]

pair_df = df[pair_cols + [LABEL_NAME]].sample(min(1000, len(df)), random_state=0)

g = sns.pairplot(pair_df, hue=LABEL_NAME, plot_kws={"alpha": 0.5, "s": 20})
g.figure.suptitle("Pairwise Sensor Scatter by Activity (subset)", y=1.02)
savefig("08_pairplot.png")

   saved eda_plots_GOH/07_correlation_heatmap.png
   saved eda_plots_GOH/08_pairplot.png


In [36]:
df["label_name"]

0          Right
1          Right
2          Right
3          Right
4          Right
           ...  
128507    Idling
128508    Idling
128509    Idling
128510    Idling
128511    Idling
Name: label_name, Length: 128512, dtype: str

In [37]:
# Z-score outlier counts
z_scores = np.abs(stats.zscore(df[sensor_cols], nan_policy="omit"))
outlier_mask = z_scores > 3

print("\nOutlier counts (|z| > 3) per channel:")
print(pd.Series(outlier_mask.sum(axis=0), index=sensor_cols))
print(f"\nTotal rows with ≥1 outlier: {outlier_mask.any(axis=1).sum()} "
      f"({outlier_mask.any(axis=1).mean():.2%})")

# Box-plots per activity
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, sensor_cols):
    sns.boxplot(data=df, x=LABEL_COL, y=col, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Sensor Distributions by Activity (box-plots)", y=1.02, fontsize=15)
plt.tight_layout()
savefig("09_boxplots_by_class.png")


Outlier counts (|z| > 3) per channel:
ACCEL_X    3217
ACCEL_Y    1529
ACCEL_Z    2764
GYRO_X      880
GYRO_Y     1442
GYRO_Z      942
dtype: int64

Total rows with ≥1 outlier: 8921 (6.94%)
   saved eda_plots_GOH/09_boxplots_by_class.png


In [38]:
window_seconds = 2
window_size = window_seconds * estimated_hz
overlap_frac = 0.5
stride = max(1, int(window_size * (1 - overlap_frac)))

print(f"\nSampling rate  : {estimated_hz} Hz")
print(f"Window length  : {window_seconds}s  →  {window_size} samples")
print(f"Overlap        : {overlap_frac:.0%}  →  stride = {stride} samples")
print(f"Input shape    : ({window_size}, {len(sensor_cols)})")

n_windows_approx = (len(df) - window_size) // stride + 1
print(f"≈ {n_windows_approx:,} windows from {len(df):,} rows")


Sampling rate  : 56 Hz
Window length  : 2s  →  112 samples
Overlap        : 50%  →  stride = 56 samples
Input shape    : (112, 6)
≈ 2,293 windows from 128,512 rows


# Results & Key Takeaways
| Item | Finding |
|---|---|
| **Rows / Columns** | `df.shape` — see cell above |
| **Sensor channels** | ACCEL_X, ACCEL_Y, ACCEL_Z, GYRO_X, GYRO_Y, GYRO_Z (6 channels) |
| **Sampling rate** | ~`estimated_hz` Hz |
| **Activities** | 9 driving-event activities (incl. aggressive variants & idling) |
| **Imbalance** | ratio printed above — consider class-weights if > 2x |
| **Missing values** | see cell 1 — handle before windowing |
| **Outliers** | z > 3 counts above — clip or robust-scale |
| **Suggested window** | 2 s @ estimated_hz → shape `(window_size, 6)` |

# **Considerations for the CNN pipeline:**
1. Impute / drop missing values.
2. Normalise each sensor channel (z-score or min-max per channel).
3. Segment into fixed-length windows with the parameters above.
4. Split windows into train / val / test **by contiguous time blocks**
    (not random) to avoid data leakage.
5. Build a 1-D CNN (Conv1D → BatchNorm → ReLU → Pool → … → Dense → Softmax).
6. Use class-weighted cross-entropy if imbalance is significant.